In [17]:
import os
from dotenv import load_dotenv
from langgraph.graph import StateGraph, add_messages, START, END
from langgraph.prebuilt import ToolNode, tools_condition
from typing import TypedDict, Annotated
from langchain_core.messages import BaseMessage, SystemMessage, HumanMessage, AIMessage

from langchain_community.vectorstores import FAISS
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_groq.chat_models import ChatGroq
from langchain_huggingface import HuggingFaceEmbeddings
from langchain.tools import tool

load_dotenv()

True

In [5]:
loader= PyPDFLoader('langchain_note.pdf')
docs= loader.load()

len(docs)

11

In [6]:
splitter= RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap= 200)
chunks= splitter.split_documents(docs)

len(chunks)

22

In [9]:
embedding= HuggingFaceEmbeddings(model_name="BAAI/bge-small-en-v1.5")

model= ChatGroq(
    model='meta-llama/llama-4-scout-17b-16e-instruct',
    api_key=os.getenv("GROQ_API_KEY")
)

d:\Users\Aditya_Kumar\LangGraphCompushX\venv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 199/199 [00:00<00:00, 5736.46it/s]


In [10]:
vector_store= FAISS.from_documents(chunks, embedding=embedding) 

retriever= vector_store.as_retriever(search_type='similarity', search_kwargs={'k':4})

In [ ]:
@tool()
def rag_tool(query):
    # this message present bz if a agent want to retrieve information from loaded document/pdf
    """
    Retrieve relevant information from the pdf document.
    Use this tool when the user asks factual/conceptual questions
    that might be answered from the stored documents.
    """

    result= retriever.invoke(query)
    # print(result)

    context= [doc.page_content for doc in result]
    metadata= [doc.metadata for doc in result]

    return {
        "query": query,
        "context": context,
        "metadata": metadata
    }

In [24]:
tools= [rag_tool]
model_with_tool= model.bind_tools(tools)

tool_node= ToolNode(tools)


class ChatState(TypedDict):
    messages: Annotated[list[BaseMessage], add_messages]


def chat_handle(state:ChatState):
    messages= [SystemMessage(content="You are question answering chat bot, and if you don't know about anything which is asked please tell that you don't know instead of telling irrelavent response"), *state['messages']]

    result= model_with_tool.invoke(messages)

    return {"messages":[result]}

builder= StateGraph(ChatState)

builder.add_node('chat_handle', chat_handle)
builder.add_node('tools', tool_node)


builder.add_edge(START, 'chat_handle')
builder.add_conditional_edges('chat_handle', tools_condition)
builder.add_edge('tools', 'chat_handle')

graph= builder.compile()

user_query= input("Ask!: ")
result= graph.invoke({"messages":[HumanMessage(content=(user_query))]})

print(result["messages"][-1].content)

[Document(id='bcfc4343-1d2a-4b98-84e1-8dfb4daebc61', metadata={'producer': 'Microsoft: Print To PDF', 'creator': 'PyPDF', 'creationdate': '2026-02-18T00:00:50+05:30', 'author': 'MysteryChap', 'moddate': '2026-02-18T00:00:50+05:30', 'title': 'Microsoft Word - langchain_article', 'source': 'langchain_note.pdf', 'total_pages': 11, 'page': 0, 'page_label': '1'}, page_content="writing bespoke glue code for every integration, developers build on LangChain's rich library of \npre-built components and composable primitives. \n \nSince its release, LangChain has grown into one of the most starred open-source AI repositories, \nbacked by a thriving ecosystem (LangSmith, LangGraph, LangServe) and enterprise adoption \nacross sectors ranging from healthcare and legal to finance and customer service. \n2. High-Level Architecture Overview \nLangChain is best understood as a layered architecture. Each layer builds on the one below it, \nproviding increasingly high-level abstractions for developers wh